In [1]:
import s3fs
import pandas as pd

# Initialize S3 filesystem (uses your AWS credentials) 
fs = s3fs.S3FileSystem(anon=False)

In [2]:
files = fs.glob("s3://collegebasketballinsiders/boxscores/game-info/*.csv") 
print(f"Found {len(files)} files") 
dfs = [] 
for path in files: 
    s3_url = f"s3://{path}" 
    try: 
        df_part = pd.read_csv(s3_url, storage_options={"anon": False}) 
        dfs.append(df_part) 
    except: 
        print(f"[skip] Could not read {s3_url}") 
    continue

game_info_df = pd.concat(dfs, ignore_index=True)

Found 27456 files


In [3]:
files = fs.glob("s3://collegebasketballinsiders/boxscores/team-stats/*.csv") 
print(f"Found {len(files)} files") 
dfs = [] 
for path in files: 
    s3_url = f"s3://{path}" 
    try: 
        df_part = pd.read_csv(s3_url, storage_options={"anon": False}) 
        dfs.append(df_part) 
    except: 
        print(f"[skip] Could not read {s3_url}") 
    continue

team_stats_df = pd.concat(dfs, ignore_index=True)

Found 27456 files


In [4]:
files = fs.glob("s3://collegebasketballinsiders/boxscores/officials/*.csv") 
print(f"Found {len(files)} files") 
dfs = [] 
for path in files: 
    s3_url = f"s3://{path}" 
    try: 
        df_part = pd.read_csv(s3_url, storage_options={"anon": False}) 
        dfs.append(df_part) 
    except: 
        print(f"[skip] Could not read {s3_url}") 
    continue

officials_df = pd.concat(dfs, ignore_index=True)

Found 27456 files


In [9]:
officials_df = (
    officials_df
    .assign(
        official_num=officials_df.groupby("game_id").cumcount() + 1
    )
    .pivot(
        index="game_id",
        columns="official_num",
        values="official_name"
    )
    .rename(columns=lambda x: f"official_{x}")
    .reset_index()
)

KeyError: 'official_name'

In [6]:
team_stats_df = (
    team_stats_df
    .pivot(
        index="game_id",
        columns="homeAway",
        values="fouls"
    )
    .rename(columns={"home": "home_fouls", "away": "away_fouls"})
    .reset_index()
)

In [7]:
game_info_df = game_info_df[["game_id", "neutral_site", "home_score", "away_score"]]

In [8]:
df = game_info_df.merge(officials_df, on="game_id",how="inner").merge(team_stats_df, on="game_id", how="inner")

In [13]:
df1 = pd.read_csv("s3://collegebasketballinsiders/officials-analysis/games.csv")

In [15]:
df2 = pd.concat([df,df1], axis=0)

In [17]:
df2.to_csv("s3://collegebasketballinsiders/officials-analysis/games.csv")

In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# ============================================================
# 1. Base data from your original df
# ============================================================

# df is your original DataFrame with game_id, scores, fouls, refs
games = df2.copy()

# Derived columns
games["total_fouls"] = games["home_fouls"] + games["away_fouls"]
games["foul_diff"] = games["home_fouls"] - games["away_fouls"]  # >0 = more on home
games["total_points"] = games["home_score"] + games["away_score"]

ref_cols = ["official_1", "official_2", "official_3", "official_4"]

# ============================================================
# 2. Referee design matrix (0/1: ref worked game)
#    Use melt() instead of set_index/stack to avoid index issues
# ============================================================

ref_long = (
    games[["game_id"] + ref_cols]
    .melt(id_vars="game_id", value_vars=ref_cols, value_name="ref")
    .dropna(subset=["ref"])
)

ref_long["value"] = 1

ref_matrix = (
    ref_long
    .pivot_table(index="game_id", columns="ref", values="value", fill_value=0)
)

# ============================================================
# 3. Filter to games with complete modeling data
# ============================================================

needed_cols = [
    "total_fouls",
    "foul_diff",
    "total_points",
    "neutral_site",
]

mask = games[needed_cols].notna().all(axis=1)
games_model = games.loc[mask].copy()

# Subset ref_matrix to those game_ids (rows) in the same order
ref_matrix_model = ref_matrix.loc[games_model["game_id"]]

# Sanity checks
assert not games_model[needed_cols].isna().any().any(), "NaNs remain in games_model."
assert not np.isnan(ref_matrix_model.values).any(), "NaNs in ref_matrix_model."

# ============================================================
# 4. Build X (controls + refs) and y targets
# ============================================================

control_cols = ["neutral_site", "total_points"]

X_controls = games_model[control_cols].astype(float).values
scaler_controls = StandardScaler()
X_controls_scaled = scaler_controls.fit_transform(X_controls)

X_refs = ref_matrix_model.values.astype(float)
X = np.hstack([X_controls_scaled, X_refs])

control_dim = X_controls_scaled.shape[1]

# Targets from *filtered* games_model
y_total_fouls = games_model["total_fouls"].values
y_diff = games_model["foul_diff"].values
y_points = games_model["total_points"].values

# Final NaN guard on y's
assert not np.isnan(y_total_fouls).any(), "NaNs in y_total_fouls."
assert not np.isnan(y_diff).any(), "NaNs in y_diff."
assert not np.isnan(y_points).any(), "NaNs in y_points."

ref_names = ref_matrix_model.columns

def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std(ddof=0)

# ============================================================
# 5. Total fouls model (whistle propensity per ref)
# ============================================================

model_total = Ridge(alpha=10.0, fit_intercept=True)
model_total.fit(X, y_total_fouls)

coef_total = model_total.coef_
coef_total_refs = coef_total[control_dim:]

ref_total_effect = pd.Series(
    coef_total_refs, index=ref_names, name="foul_effect_model"
)
ref_total_effect_z = zscore(ref_total_effect).rename("foul_propensity_z")

# ============================================================
# 6. Home / away bias model
#    Target = home_fouls - away_fouls
# ============================================================

model_diff = Ridge(alpha=10.0, fit_intercept=True)
model_diff.fit(X, y_diff)

coef_diff = model_diff.coef_
coef_diff_refs = coef_diff[control_dim:]

ref_home_minus_away = pd.Series(
    coef_diff_refs, index=ref_names, name="home_minus_away_effect"
)

# Positive = more fouls on AWAY team (home-friendly)
ref_home_bias_index = (-ref_home_minus_away).rename("home_bias_index")
ref_home_bias_z = zscore(ref_home_bias_index).rename("home_bias_z")

# ============================================================
# 7. Points environment model
#    Target = total_points
# ============================================================

model_pts = Ridge(alpha=10.0, fit_intercept=True)
model_pts.fit(X, y_points)

coef_pts = model_pts.coef_
coef_pts_refs = coef_pts[control_dim:]

ref_points_effect = pd.Series(
    coef_pts_refs, index=ref_names, name="points_effect_model"
)
ref_points_z = zscore(ref_points_effect).rename("points_effect_z")

# ============================================================
# 8. Raw per-ref stats for context
# ============================================================

ref_game_long = (
    games_model[["game_id"] + ref_cols]
    .melt(id_vars="game_id", value_vars=ref_cols, value_name="ref")
    .dropna(subset=["ref"])
)

metrics = games_model[
    ["game_id", "total_fouls", "home_fouls", "away_fouls",
     "foul_diff", "total_points", "neutral_site"]
]

ref_game_long = ref_game_long.merge(metrics, on="game_id", how="left")

ref_raw_stats = (
    ref_game_long.groupby("ref")
    .agg(
        games_officiated=("game_id", "nunique"),
        mean_total_fouls=("total_fouls", "mean"),
        mean_home_fouls=("home_fouls", "mean"),
        mean_away_fouls=("away_fouls", "mean"),
        mean_foul_diff=("foul_diff", "mean"),
        mean_total_points=("total_points", "mean"),
        neutral_site_share=("neutral_site", "mean"),
    )
)

# ============================================================
# 9. Combine modeled effects + raw stats into final table
# ============================================================

ref_effects = pd.concat(
    [
        ref_total_effect,
        ref_total_effect_z,
        ref_home_minus_away,
        ref_home_bias_index,
        ref_home_bias_z,
        ref_points_effect,
        ref_points_z,
    ],
    axis=1,
)

ref_summary = ref_raw_stats.join(ref_effects, how="left")

# Optional: only include refs with a minimum number of games
min_games = 10
ref_summary_filtered = ref_summary[ref_summary["games_officiated"] >= min_games]

# Example: top whistle-happy refs
print(ref_summary_filtered.sort_values("foul_propensity_z", ascending=False).head())


                     games_officiated  mean_total_fouls  mean_home_fouls  \
ref                                                                        
Jon Wall                           48         39.666667        19.500000   
Ronald Brokenbrough                58         39.655172        19.379310   
Robert Lehigh                      59         39.779661        19.271186   
John Gleich                        20         40.100000        20.050000   
Jon Stigliano                     104         39.750000        19.278846   

                     mean_away_fouls  mean_foul_diff  mean_total_points  \
ref                                                                       
Jon Wall                   20.166667       -0.666667         142.395833   
Ronald Brokenbrough        20.275862       -0.896552         144.568966   
Robert Lehigh              20.508475       -1.237288         145.152542   
John Gleich                20.050000        0.000000         140.300000   
Jon Stigliano    

In [19]:
ref_final = ref_summary_filtered.reset_index()[["ref", "games_officiated","foul_propensity_z", "home_bias_z","points_effect_z"]]

In [20]:

# Columns to rank (you can add/remove as needed)
rank_cols = [
    "foul_propensity_z",
    "home_bias_z",
    "points_effect_z",
]

# Create rank columns
for col in rank_cols:
    # rank: highest value gets rank 1 (descending=True)
    rank_col_name = col + "_rank"
    ref_final[rank_col_name] = ref_final[col].rank(ascending=False, method="dense")

In [21]:
ref_final.sort_values("foul_propensity_z_rank", ascending=True)

,ref,games_officiated,foul_propensity_z,home_bias_z,points_effect_z,foul_propensity_z_rank,home_bias_z_rank,points_effect_z_rank
391,Jon Wall,48,3.909950,-0.621924,-0.874436,1.0,562.0,604.0
631,Ronald Brokenbrough,58,3.781921,-0.666431,0.041086,2.0,575.0,342.0
620,Robert Lehigh,59,3.720998,-0.060505,0.521953,3.0,379.0,221.0
372,John Gleich,20,3.491117,-1.588840,-1.321419,4.0,705.0,685.0
389,Jon Stigliano,104,3.476183,0.004513,0.401536,5.0,363.0,249.0
...,...,...,...,...,...,...,...,...
234,Evan Berg,65,-2.882659,-0.930441,0.552018,753.0,625.0,213.0
319,Javed Trotman,26,-2.897337,-0.376488,1.181883,754.0,490.0,103.0
293,Jake Kuhlman,19,-3.054187,1.597484,0.329425,755.0,43.0,269.0
343,Jeremy Trussell,42,-3.888334,0.986645,0.662856,756.0,115.0,187.0


In [22]:
ref_final.to_csv("s3://collegebasketballinsiders/officials-analysis/officials.csv")